In [1]:
import heapq
import numpy as np
import pandas as pd
import plotly.express as px
import collections
import itertools
import tqdm
from copy import deepcopy


In [2]:
np.random.seed(0)

In [3]:
def normalize_priors(priors):
    if np.sum(priors) == 0:
        return np.zeros_like(priors)
    return priors / np.sum(priors)

In [4]:
def best_response(x, thresholds, priors, c):
    posteriors = normalize_priors(priors)
    search_space = [x] + thresholds[thresholds > x].tolist()
    utilities = []
    for x_p in search_space:
        utility = np.dot(posteriors, x_p >= thresholds)
        cost = c * abs(x-x_p)
        utilities.append(utility - cost)
    return search_space[np.argmax(utilities)]

def best_response_vectorized(X, thresholds, priors, c):
    posteriors = normalize_priors(priors).flatten()
    utilities_expected = np.array([
        np.dot(posteriors, thresholds[j] >= thresholds)
        for j in range(len(thresholds))
    ])

    pass_matrix = X[:, None] >= thresholds[None, :]
    utility_stay = pass_matrix @ posteriors

    X_col = X[:, None]
    thresholds_row = thresholds[None, :]

    feasible = thresholds_row > X_col
    utility_jump = utilities_expected[None, :] - c * np.abs(thresholds_row - X_col)
    utility_jump[~feasible] = -np.inf

    utilities = np.concatenate([utility_stay[:, None], utility_jump],axis=1)

    best_idx = np.argmax(utilities, axis=1)
    X_p = np.where(best_idx == 0, X, thresholds[best_idx - 1])

    return X_p

In [5]:
def accuracy_loss_vectorized(X, X_p, thresholds, priors, threshold_true):
    Y_true = (X >= threshold_true).astype(float)
    Y_p = (X_p[:, None] >= thresholds[None, :]).astype(float)
    losses = np.abs(Y_true[:, None] - Y_p).mean(axis=0)
    posteriors = normalize_priors(priors)
    return np.dot(losses, posteriors)

In [6]:
def evaluate_partition(X, partition, thresholds, priors, threshold_true, c):
    thresholds_p = thresholds[partition]
    priors_p     = priors[partition]
    X_p = best_response_vectorized(X, thresholds_p, priors_p, c)
    return accuracy_loss_vectorized(X, X_p, thresholds_p, priors_p, threshold_true)

def evaluate_system(X, partitions, thresholds, priors, threshold_true, c):
    acc_loss = 0.0
    for partition in partitions:
        acc_loss_p  = evaluate_partition(X, partition, thresholds, priors, threshold_true, c)
        acc_loss += acc_loss_p * np.sum(priors[partition])
    return acc_loss

In [7]:
def set_partitions(collection):
    if len(collection) == 1:
        yield [collection]
        return

    first = collection[0]
    for smaller in set_partitions(collection[1:]):
        for i in range(len(smaller)):
            yield smaller[:i] + [[first] + smaller[i]] + smaller[i+1:]
        yield [[first]] + smaller

In [8]:
def find_partitions_optimal(X, thresholds, priors, threshold_true, c):
    indices = list(range(len(thresholds)))
    partitions_set = list(set_partitions(indices))
    best_partition, best_loss = None, np.inf
    for partitions in partitions_set:
        acc_loss = evaluate_system(X, partitions, thresholds, priors, threshold_true, c)
        if acc_loss < best_loss:
            best_loss      = acc_loss
            best_partition = partitions
    return best_partition

In [9]:
def display_priority_queue(pq, P):
    res = "[  "
    for acc_loss, (a_id, b_id) in pq:
        res += f"({acc_loss:.4f}, ({P[a_id]}, {P[b_id]}))  "
    res += "]"
    print(res)

def find_partitions_greedy_best(X, thresholds, priors, threshold_true, c, display=False):
    P = {}
    next_id = 0
    partitions = [[i] for i in range(len(priors))]
    for block in partitions:
        P[next_id] = list(block)
        next_id += 1
    
    pq = []
    for a_id, b_id in itertools.combinations(P.keys(), 2):
        a, b = P[a_id], P[b_id]
        ab = sorted(a+b)
        acc_loss_ab = evaluate_partition(X, ab, thresholds, priors, threshold_true, c) * np.sum(priors[ab])
        acc_loss_a = evaluate_partition(X, a, thresholds, priors, threshold_true, c) * np.sum(priors[a])
        acc_loss_b = evaluate_partition(X, b, thresholds, priors, threshold_true, c) * np.sum(priors[b])
        gain = -(acc_loss_a + acc_loss_b - acc_loss_ab)
        heapq.heappush(pq, (gain, (a_id, b_id)))

    while pq:
        if display:
            display_priority_queue(pq, P)
        gain, (a_id, b_id) = heapq.heappop(pq)
        if a_id not in P.keys() or b_id not in P.keys():
            continue

        a = P[a_id]
        b = P[b_id]

        ab = sorted(a + b)
        acc_loss_a = evaluate_partition(X, a, thresholds, priors, threshold_true, c)
        acc_loss_b = evaluate_partition(X, b, thresholds, priors, threshold_true, c)
        acc_loss_ab = evaluate_partition(X, ab, thresholds, priors, threshold_true, c)
        lhs = acc_loss_a * np.sum(priors[a]) + acc_loss_b * np.sum(priors[b])
        rhs = acc_loss_ab * np.sum(priors[ab])

        if lhs - rhs > -1e-6:
            del P[a_id]
            del P[b_id]

            pq2 = []
            for acc_loss, (x_id, y_id) in pq:
                if x_id not in {a_id, b_id} and y_id not in {a_id, b_id}:
                    heapq.heappush(pq2, (acc_loss, (x_id, y_id)))
            pq = deepcopy(pq2)

            new_id = next_id
            next_id += 1
            P[new_id] = ab

            for p_id in P.keys():
                if p_id != new_id:
                    p = P[p_id]
                    merged = sorted(ab + p)
                    acc_loss_merged = evaluate_partition(X, merged, thresholds, priors, threshold_true, c) * np.sum(priors[merged])
                    acc_loss_p = evaluate_partition(X, p, thresholds, priors, threshold_true, c) * np.sum(priors[p])
                    acc_loss_ab = evaluate_partition(X, ab, thresholds, priors, threshold_true, c) * np.sum(priors[ab])
                    gain = -(acc_loss_p + acc_loss_ab - acc_loss_merged)
                    heapq.heappush(pq, (gain, (new_id, p_id)))
    return list(P.values())

In [10]:
def approximation_ratio(loss_optimal, loss_greedy, rtype="m"):
    if rtype in ["a", "add", "additive"]:
        return loss_greedy - loss_optimal
    elif rtype in ["m", "mult", "multiplicative"]:
        if loss_optimal == 0:
            return np.nan
        return loss_greedy / loss_optimal

In [11]:
x_min, x_max, x_disc = 0., 1., 1e-4
X = np.arange(x_min, x_max+x_disc, x_disc).round(4)

In [38]:
df = pd.read_pickle("../results/grid_search_approx_ratio_best_first_n3.pkl")

In [26]:
df.head()

,threshold_true,c,partition_opt,loss_opt,partition_greedy,loss_greedy,r_mult,r_add,thresholds,priors
0,0.1,0.75,"[[0, 1, 2]]",0.09999,"[[0, 1, 2]]",0.09999,1.0,0.0,"[0.0, 0.05, 0.1]","[1.0, 0.0, 0.0]"
1,0.2,0.75,"[[0, 1, 2]]",0.19998,"[[0, 1, 2]]",0.19998,1.0,0.0,"[0.0, 0.05, 0.1]","[1.0, 0.0, 0.0]"
2,0.3,0.75,"[[0, 1, 2]]",0.29997,"[[0, 1, 2]]",0.29997,1.0,0.0,"[0.0, 0.05, 0.1]","[1.0, 0.0, 0.0]"
3,0.4,0.75,"[[0, 1, 2]]",0.39996,"[[0, 1, 2]]",0.39996,1.0,0.0,"[0.0, 0.05, 0.1]","[1.0, 0.0, 0.0]"
4,0.5,0.75,"[[0, 1, 2]]",0.49995,"[[0, 1, 2]]",0.49995,1.0,0.0,"[0.0, 0.05, 0.1]","[1.0, 0.0, 0.0]"


In [27]:
i = df["r_mult"].argmax()
df.iloc[[i]]

,threshold_true,c,partition_opt,loss_opt,partition_greedy,loss_greedy,r_mult,r_add,thresholds,priors
2922,0.3,0.75,"[[1], [0, 2]]",0.29997,"[[0, 1, 2]]",0.29997,1.0,1.110223e-16,"[0.0, 0.05, 0.1]","[0.54, 0.14, 0.32]"


In [28]:
thresholds = df["thresholds"].iloc[i]
priors = df["priors"].iloc[i]
threshold_true = df["threshold_true"].iloc[i]
c = df["c"].iloc[i]
partition_opt = df["partition_opt"].iloc[i]
partition_greedy = df["partition_greedy"].iloc[i]
loss_opt = df["loss_opt"].iloc[i]
loss_greedy = df["loss_greedy"].iloc[i]
r = df["r_mult"].iloc[i]

In [29]:
display(pd.DataFrame({"Thresholds": thresholds, "Priors": priors}).round(6).T)
print(f"c              : {c}")
print(f"t*             : {threshold_true:.4f}")
print()
print("Greedy")
print("------")
print(f"Partition      : {partition_greedy}")
print(f"Accuracy loss  : {loss_greedy:.6f}")
print()
print("Optimal")
print("-------")
print(f"Partition      : {partition_opt}")
print(f"Accuracy loss  : {loss_opt:.6f}")
print()
print(f"r (mult)       : {r:.4f}")
print("-"*80)
print()

,0,1,2
Thresholds,0.00,0.05,0.10
Priors,0.54,0.14,0.32


c              : 0.75
t*             : 0.3000

Greedy
------
Partition      : [[0, 1, 2]]
Accuracy loss  : 0.299970

Optimal
-------
Partition      : [[1], [0, 2]]
Accuracy loss  : 0.299970

r (mult)       : 1.0000
--------------------------------------------------------------------------------



In [37]:
a, b = [0], [1]

acc_loss_b = evaluate_partition(X, b, thresholds, priors, threshold_true, c)
acc_loss_a = evaluate_partition(X, a, thresholds, priors, threshold_true, c)

lhs = acc_loss_a * np.sum(priors[a]) + acc_loss_b * np.sum(priors[b])
ab = sorted(a + b)

acc_loss_ab = evaluate_partition(X, ab, thresholds, priors, threshold_true, c)
rhs = acc_loss_ab * np.sum(priors[ab])
gain = lhs - rhs

print(f"    threshold true: {threshold_true:.4f}")
print(f"                 c: {c}")
print(f"                 a: {a}")
print(f"                 b: {b}")
print(f"   accuracy loss a: {acc_loss_a:.7f}")
print(f"   accuracy loss b: {acc_loss_b:.7f}")
print(f"  accuracy loss ab: {acc_loss_ab:.7f}")
print(f"               LHS: {lhs:.7f}")
print(f"               RHS: {rhs:.7f}")
print(f"              gain: {gain:.7f}")
print(f"            merge?: {lhs - rhs > -1e-6}")
print()

    threshold true: 0.3000
                 c: 0.75
                 a: [0]
                 b: [1]
   accuracy loss a: 0.2999700
   accuracy loss b: 0.2999700
  accuracy loss ab: 0.2999700
               LHS: 0.2039796
               RHS: 0.2039796
              gain: 0.0000000
            merge?: True



In [32]:
worst_r = 1. 
worst_c = 0.748
for c_ in np.arange(0.748, 0.758, 1e-5).round(5):
    partition_opt = find_partitions_optimal(X, thresholds, priors, threshold_true, c_)
    partition_greedy = find_partitions_greedy_best(X, thresholds, priors, threshold_true, c_)

    loss_opt = evaluate_system(X, partition_opt, thresholds, priors, threshold_true, c_)
    loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors, threshold_true, c_)

    r = approximation_ratio(loss_opt, loss_greedy)
    if r > worst_r:
        worst_r = r
        worst_c = c_

# worst_c = 0.75
partition_opt = find_partitions_optimal(X, thresholds, priors, threshold_true, worst_c)
partition_greedy = find_partitions_greedy_best(X, thresholds, priors, threshold_true, worst_c)
loss_opt = evaluate_system(X, partition_opt, thresholds, priors, threshold_true, worst_c)
loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors, threshold_true, worst_c)
r = approximation_ratio(loss_opt, loss_greedy)

In [34]:
display(pd.DataFrame({"Thresholds": thresholds, "Priors": priors}).round(6).T)
print(f"c              : {worst_c}")
print(f"t*             : {threshold_true:.4f}")
print()
print("Greedy")
print("------")
print(f"Partition      : {partition_greedy}")
print(f"Accuracy loss  : {loss_greedy:.6f}")
print()
print("Optimal")
print("-------")
print(f"Partition      : {partition_opt}")
print(f"Accuracy loss  : {loss_opt:.6f}")
print()
print(f"r (mult)       : {r:.4f}")
print("-"*80)
print()

,0,1,2
Thresholds,0.00,0.05,0.10
Priors,0.54,0.14,0.32


c              : 0.748
t*             : 0.3000

Greedy
------
Partition      : [[0, 1, 2]]
Accuracy loss  : 0.299970

Optimal
-------
Partition      : [[0], [1, 2]]
Accuracy loss  : 0.299970

r (mult)       : 1.0000
--------------------------------------------------------------------------------

